# Colab 3: Sudoku + Zebra Puzzle Experiments
## Reproducing Tables 2, 3, and 5

**Experiments**:
- **Table 2**: Sudoku accuracy — ARM (w/wo ordering) vs MDM (vanilla/top-prob/top-prob-margin)
- **Table 3**: Zebra accuracy — same comparison
- **Table 5**: Hard Sudoku generalization — same models, hard test set

**Models**:
- Sudoku MDM: 6M params, GPT-2 architecture
- Zebra MDM: 19M params
- ARM baselines: 42M params each

**Training** (Appendix D.2): lr=0.001, batch_size=128, 300 epochs

**Inference**: 50 reverse steps, Gumbel noise coefficient 0.5

**Runtime**: A100 GPU, ~24-48 hours

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'

!pip install torch torchvision transformers einops tqdm matplotlib -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
import math
import pickle
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

---
## Cell 2: Model Architectures (MDM + ARM)

**MDM**: Bidirectional attention, no time embedding, RoPE  
**ARM**: Causal attention, standard autoregressive  

Both follow GPT-2 architecture. Sizes:
- 6M: hidden=256, layers=6, heads=4, ff=1024
- 19M: hidden=384, layers=8, heads=6, ff=1536
- 42M: hidden=512, layers=10, heads=8, ff=2048

In [ ]:
class RoPE(nn.Module):
    def __init__(self, dim, max_seq_len=512):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
    
    def forward(self, x, seq_len):
        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1, causal=False):
        super().__init__()
        self.causal = causal
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim), nn.GELU(),
            nn.Linear(ff_dim, hidden_dim), nn.Dropout(dropout)
        )
        self.attn_dropout = nn.Dropout(dropout)
    
    def forward(self, x, rope_cos=None, rope_sin=None):
        B, L, D = x.shape
        h = self.ln1(x)
        qkv = self.qkv(h).reshape(B, L, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if rope_cos is not None:
            q, k = apply_rotary_pos_emb(q, k, rope_cos, rope_sin)
        attn = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if self.causal:
            mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
            attn = attn.masked_fill(mask[None, None], float('-inf'))
        attn = self.attn_dropout(F.softmax(attn, dim=-1))
        out = self.out_proj(torch.matmul(attn, v).transpose(1, 2).reshape(B, L, D))
        x = x + out
        return x + self.ff(self.ln2(x))


class MDMTransformer(nn.Module):
    """Bidirectional transformer for MDM. No time embedding."""
    def __init__(self, vocab_size, hidden_dim, num_layers, num_heads, ff_dim,
                 max_seq_len=128, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.rope = RoPE(hidden_dim // num_heads, max_seq_len)
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads, ff_dim, dropout, causal=False)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(hidden_dim)
        self.output_head = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        B, L = x.shape
        h = self.embedding(x)
        rope_cos, rope_sin = self.rope(h, L)
        for block in self.blocks:
            h = block(h, rope_cos, rope_sin)
        return self.output_head(self.ln_final(h))


class ARMTransformer(nn.Module):
    """Causal transformer for ARM."""
    def __init__(self, vocab_size, hidden_dim, num_layers, num_heads, ff_dim,
                 max_seq_len=128, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_emb = nn.Embedding(max_seq_len, hidden_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads, ff_dim, dropout, causal=True)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(hidden_dim)
        self.output_head = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        B, L = x.shape
        h = self.embedding(x) + self.pos_emb(torch.arange(L, device=x.device))
        for block in self.blocks:
            h = block(h)
        return self.output_head(self.ln_final(h))


# Model configs
CONFIGS = {
    '6M':  {'hidden_dim': 256, 'num_layers': 6,  'num_heads': 4, 'ff_dim': 1024},
    '19M': {'hidden_dim': 384, 'num_layers': 8,  'num_heads': 6, 'ff_dim': 1536},
    '42M': {'hidden_dim': 512, 'num_layers': 10, 'num_heads': 8, 'ff_dim': 2048},
}

# Sudoku vocab: 0=mask, 1-9=digits => vocab_size=10
SUDOKU_VOCAB = 10
SUDOKU_SEQ_LEN = 81
SUDOKU_MASK = 0

print("Architectures defined.")
for name, cfg in CONFIGS.items():
    m = MDMTransformer(SUDOKU_VOCAB, max_seq_len=SUDOKU_SEQ_LEN, **cfg)
    print(f"  {name}: {sum(p.numel() for p in m.parameters())/1e6:.1f}M params")

---
## Cell 3: Training Functions

In [ ]:
def mdm_train_step(model, optimizer, solutions, mask_token_id, device):
    """MDM training: mask random positions, predict original tokens."""
    model.train()
    B, L = solutions.shape
    
    n = torch.randint(1, L + 1, (B,), device=device)
    noise = torch.rand(B, L, device=device)
    sorted_idx = noise.argsort(dim=-1)
    
    mask = torch.zeros(B, L, dtype=torch.bool, device=device)
    for b in range(B):
        mask[b, sorted_idx[b, :n[b]]] = True
    
    x_masked = solutions.clone()
    x_masked[mask] = mask_token_id
    
    logits = model(x_masked)
    
    # 1/n weighted loss
    total_loss = torch.tensor(0.0, device=device)
    for b in range(B):
        if mask[b].sum() > 0:
            sample_loss = F.cross_entropy(logits[b, mask[b]], solutions[b, mask[b]], reduction='sum')
            total_loss = total_loss + sample_loss / n[b].float()
    total_loss = total_loss / B
    
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return total_loss.item()


def arm_train_step(model, optimizer, sequences, device):
    """Standard causal LM training."""
    model.train()
    logits = model(sequences[:, :-1])
    targets = sequences[:, 1:]
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return loss.item()


def train_model(model, train_fn, data_loader_fn, num_epochs, lr=0.001,
                device='cuda', save_path=None, log_interval=10):
    """
    Generic training loop.
    Appendix D.2: lr=0.001, batch_size=128, 300 epochs.
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    model = model.to(device)
    
    all_losses = []
    for epoch in range(num_epochs):
        epoch_losses = []
        for batch in data_loader_fn():
            batch = batch.to(device)
            loss = train_fn(model, optimizer, batch, device)
            epoch_losses.append(loss)
        
        avg = np.mean(epoch_losses)
        all_losses.append(avg)
        if (epoch + 1) % log_interval == 0:
            print(f"  Epoch {epoch+1}/{num_epochs}, Loss: {avg:.4f}")
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(model.state_dict(), save_path)
        print(f"  Saved to {save_path}")
    return all_losses

print("Training functions defined.")

---
## Cell 4: Data Loading

In [ ]:
# Load Sudoku data
sudoku_train_clues = np.load(f'{BASE_DIR}/data/sudoku/train/clues.npy')
sudoku_train_solutions = np.load(f'{BASE_DIR}/data/sudoku/train/solutions.npy')
sudoku_test_easy_clues = np.load(f'{BASE_DIR}/data/sudoku/test_easy/clues.npy')
sudoku_test_easy_solutions = np.load(f'{BASE_DIR}/data/sudoku/test_easy/solutions.npy')
sudoku_test_hard_clues = np.load(f'{BASE_DIR}/data/sudoku/test_hard/clues.npy')
sudoku_test_hard_solutions = np.load(f'{BASE_DIR}/data/sudoku/test_hard/solutions.npy')

with open(f'{BASE_DIR}/data/sudoku/train/solve_orders.pkl', 'rb') as f:
    sudoku_solve_orders = pickle.load(f)

print(f"Sudoku train: {sudoku_train_solutions.shape}")
print(f"Sudoku test easy: {sudoku_test_easy_solutions.shape}")
print(f"Sudoku test hard: {sudoku_test_hard_solutions.shape}")
print(f"Solve orders: {len(sudoku_solve_orders)} entries")

BATCH_SIZE = 128

def make_sudoku_mdm_loader():
    """Yield batches of complete Sudoku solutions for MDM training."""
    indices = np.random.permutation(len(sudoku_train_solutions))
    for i in range(0, len(indices) - BATCH_SIZE + 1, BATCH_SIZE):
        batch_idx = indices[i:i+BATCH_SIZE]
        yield torch.tensor(sudoku_train_solutions[batch_idx], dtype=torch.long)

def make_sudoku_arm_loader(with_ordering=False):
    """Yield batches for ARM training (optionally with solve ordering)."""
    indices = np.random.permutation(len(sudoku_train_solutions))
    for i in range(0, len(indices) - BATCH_SIZE + 1, BATCH_SIZE):
        batch_idx = indices[i:i+BATCH_SIZE]
        if with_ordering:
            # Reorder each sequence according to its solve order:
            # Given clues come first (in original order), then empty cells in solve order
            batch = []
            for j in batch_idx:
                clues = sudoku_train_clues[j]
                solution = sudoku_train_solutions[j]
                order = sudoku_solve_orders[j]
                
                # Clue positions first
                clue_positions = [pos for pos in range(81) if clues[pos] != 0]
                # Solve-order positions next
                solved_positions = [r * 9 + c for r, c, d in order]
                # Any remaining (shouldn't happen if solve_order is complete)
                all_used = set(clue_positions) | set(solved_positions)
                remaining = [pos for pos in range(81) if pos not in all_used]
                
                ordered_positions = clue_positions + solved_positions + remaining
                reordered = solution[ordered_positions]
                batch.append(reordered)
            yield torch.tensor(np.array(batch), dtype=torch.long)
        else:
            yield torch.tensor(sudoku_train_solutions[batch_idx], dtype=torch.long)

print("Data loaders ready.")

---
## Cell 5: MDM Inference for Puzzles (Conditional Generation)

In [ ]:
@torch.no_grad()
def puzzle_mdm_inference(model, clues, mask_token_id=0, vocab_size=10,
                         strategy='vanilla', num_steps=50, gumbel_coeff=0.5,
                         device='cuda'):
    """
    Conditional MDM inference for puzzles.
    
    Given clues are FIXED throughout inference.
    Only masked (empty) positions are predicted.
    
    Args:
        clues: (81,) numpy array, 0=empty, 1-9=given digits
    Returns:
        (81,) tensor with all positions filled
    """
    model.eval()
    seq_len = len(clues)
    
    x = torch.tensor(clues, dtype=torch.long, device=device).unsqueeze(0)  # (1, 81)
    fixed_mask = (x[0] != mask_token_id)  # True for given clue positions
    
    timesteps = torch.linspace(1.0, 0.0, num_steps + 1)
    
    for step in range(num_steps):
        t = timesteps[step].item()
        s = timesteps[step + 1].item()
        
        # Masked positions that are not fixed clues
        is_masked = (x[0] == mask_token_id) & ~fixed_mask
        masked_positions = is_masked.nonzero(as_tuple=True)[0]
        num_masked = len(masked_positions)
        if num_masked == 0:
            break
        
        # K = num_masked * (t - s) / t
        K = max(1, round(num_masked * (t - s) / (t + 1e-10)))
        K = min(K, num_masked)
        
        logits = model(x)  # (1, 81, vocab_size)
        masked_logits = logits[0, masked_positions]  # (num_masked, vocab_size)
        masked_logits[:, mask_token_id] = float('-inf')  # Exclude mask token
        probs = F.softmax(masked_logits, dim=-1)
        
        if strategy == 'vanilla':
            perm = torch.randperm(num_masked, device=device)[:K]
            for idx in perm:
                token = torch.multinomial(probs[idx], 1).item()
                x[0, masked_positions[idx]] = token
        
        elif strategy == 'top_prob':
            scores = probs.max(dim=-1).values
            if gumbel_coeff > 0:
                gumbel = -torch.log(-torch.log(torch.rand_like(scores) + 1e-8) + 1e-8)
                scores = scores + gumbel_coeff * gumbel
            _, top_k = torch.topk(scores, K)
            for idx in top_k:
                token = torch.multinomial(probs[idx], 1).item()
                x[0, masked_positions[idx]] = token
        
        elif strategy == 'top_prob_margin':
            top2_vals, _ = torch.topk(probs, k=min(2, probs.shape[-1]), dim=-1)
            scores = top2_vals[:, 0] - top2_vals[:, 1] if top2_vals.shape[-1] >= 2 else top2_vals[:, 0]
            if gumbel_coeff > 0:
                gumbel = -torch.log(-torch.log(torch.rand_like(scores) + 1e-8) + 1e-8)
                scores = scores + gumbel_coeff * gumbel
            _, top_k = torch.topk(scores, K)
            for idx in top_k:
                token = torch.multinomial(probs[idx], 1).item()
                x[0, masked_positions[idx]] = token
    
    # Fill remaining
    remaining = (x[0] == mask_token_id) & ~fixed_mask
    if remaining.any():
        logits = model(x)
        for pos in remaining.nonzero(as_tuple=True)[0]:
            logits[0, pos, mask_token_id] = float('-inf')
            x[0, pos] = torch.multinomial(F.softmax(logits[0, pos], dim=-1), 1).item()
    
    return x[0]


@torch.no_grad()
def arm_inference(model, clues=None, seq_len=81, vocab_size=10, device='cuda'):
    """Standard left-to-right ARM generation."""
    model.eval()
    x = torch.zeros(1, seq_len, dtype=torch.long, device=device)
    if clues is not None:
        x[0] = torch.tensor(clues, dtype=torch.long, device=device)
    
    # Generate left-to-right, skipping given clues
    for pos in range(seq_len):
        if clues is not None and clues[pos] != 0:
            continue
        logits = model(x[:, :pos+1])
        probs = F.softmax(logits[0, -1, 1:], dim=-1)  # Exclude token 0
        token = torch.multinomial(probs, 1).item() + 1
        x[0, pos] = token
    return x[0]

print("Inference functions defined.")

---
## Cell 6: Train Sudoku Models

In [ ]:
# ---- 6M MDM for Sudoku ----
print("=== Training 6M Sudoku MDM ===")
sudoku_mdm = MDMTransformer(
    vocab_size=SUDOKU_VOCAB, max_seq_len=SUDOKU_SEQ_LEN, **CONFIGS['6M']
).to(device)
print(f"Params: {sum(p.numel() for p in sudoku_mdm.parameters())/1e6:.1f}M")

mdm_train_fn = lambda model, opt, batch, dev: mdm_train_step(model, opt, batch, SUDOKU_MASK, dev)

train_model(
    sudoku_mdm, mdm_train_fn, make_sudoku_mdm_loader,
    num_epochs=300, lr=0.001, device=device,
    save_path=f'{BASE_DIR}/models/sudoku_mdm_6M.pt'
)

In [ ]:
# ---- 42M ARM without ordering ----
print("\n=== Training 42M Sudoku ARM (without ordering) ===")
sudoku_arm_no_order = ARMTransformer(
    vocab_size=SUDOKU_VOCAB, max_seq_len=SUDOKU_SEQ_LEN, **CONFIGS['42M']
).to(device)
print(f"Params: {sum(p.numel() for p in sudoku_arm_no_order.parameters())/1e6:.1f}M")

arm_no_order_fn = lambda model, opt, batch, dev: arm_train_step(model, opt, batch, dev)

train_model(
    sudoku_arm_no_order, arm_no_order_fn,
    lambda: make_sudoku_arm_loader(with_ordering=False),
    num_epochs=300, lr=0.001, device=device,
    save_path=f'{BASE_DIR}/models/sudoku_arm_42M_no_order.pt'
)

In [ ]:
# ---- 42M ARM with ordering (Shah et al. 2024 baseline) ----
print("\n=== Training 42M Sudoku ARM (with ordering) ===")
sudoku_arm_with_order = ARMTransformer(
    vocab_size=SUDOKU_VOCAB, max_seq_len=SUDOKU_SEQ_LEN, **CONFIGS['42M']
).to(device)
print(f"Params: {sum(p.numel() for p in sudoku_arm_with_order.parameters())/1e6:.1f}M")

train_model(
    sudoku_arm_with_order, arm_no_order_fn,
    lambda: make_sudoku_arm_loader(with_ordering=True),
    num_epochs=300, lr=0.001, device=device,
    save_path=f'{BASE_DIR}/models/sudoku_arm_42M_with_order.pt'
)

---
## Cell 7: Evaluate Sudoku — Table 2 (Easy) and Table 5 (Hard)

In [ ]:
def evaluate_sudoku(model, test_clues, test_solutions, strategy, model_type='mdm',
                    num_eval=1000, num_steps=50, gumbel_coeff=0.5):
    """
    Evaluate Sudoku puzzle accuracy.
    A puzzle is correct IFF ALL 81 cells match the solution.
    """
    correct = 0
    num_eval = min(num_eval, len(test_clues))
    
    for i in tqdm(range(num_eval), desc=f"Eval {strategy}"):
        clues = test_clues[i]
        solution = test_solutions[i]
        
        if model_type == 'mdm':
            pred = puzzle_mdm_inference(
                model, clues, mask_token_id=SUDOKU_MASK,
                vocab_size=SUDOKU_VOCAB, strategy=strategy,
                num_steps=num_steps, gumbel_coeff=gumbel_coeff,
                device=device
            ).cpu().numpy()
        elif model_type == 'arm':
            pred = arm_inference(
                model, clues=clues, seq_len=81,
                vocab_size=SUDOKU_VOCAB, device=device
            ).cpu().numpy()
        
        if np.array_equal(pred, solution):
            correct += 1
    
    return correct / num_eval


# ---- Table 2: Easy Sudoku ----
print("=" * 60)
print("TABLE 2: Sudoku Accuracy (Easy Test Set)")
print("=" * 60)

table2_results = {}
NUM_EVAL = 1000

# Load models if not in memory
sudoku_mdm.load_state_dict(torch.load(f'{BASE_DIR}/models/sudoku_mdm_6M.pt', map_location=device))

for strategy in ['vanilla', 'top_prob', 'top_prob_margin']:
    acc = evaluate_sudoku(
        sudoku_mdm, sudoku_test_easy_clues, sudoku_test_easy_solutions,
        strategy=strategy, model_type='mdm', num_eval=NUM_EVAL
    )
    table2_results[f'MDM_{strategy}'] = acc
    print(f"  MDM ({strategy}): {acc*100:.2f}%")

# ARM without ordering
sudoku_arm_no_order.load_state_dict(torch.load(f'{BASE_DIR}/models/sudoku_arm_42M_no_order.pt', map_location=device))
acc = evaluate_sudoku(
    sudoku_arm_no_order, sudoku_test_easy_clues, sudoku_test_easy_solutions,
    strategy='vanilla', model_type='arm', num_eval=NUM_EVAL
)
table2_results['ARM_no_order'] = acc
print(f"  ARM (w/o ordering): {acc*100:.2f}%")

# ARM with ordering
sudoku_arm_with_order.load_state_dict(torch.load(f'{BASE_DIR}/models/sudoku_arm_42M_with_order.pt', map_location=device))
acc = evaluate_sudoku(
    sudoku_arm_with_order, sudoku_test_easy_clues, sudoku_test_easy_solutions,
    strategy='vanilla', model_type='arm', num_eval=NUM_EVAL
)
table2_results['ARM_with_order'] = acc
print(f"  ARM (with ordering): {acc*100:.2f}%")

with open(f'{BASE_DIR}/results/table2.json', 'w') as f:
    json.dump(table2_results, f, indent=2)

print("\nExpected (paper):")
print("  ARM (w/o ordering): 9.73%")
print("  ARM (with ordering): 87.18%")
print("  MDM vanilla: 6.88%")
print("  MDM top_prob: 18.51%")
print("  MDM top_prob_margin: 89.49%")

In [ ]:
# ---- Table 5: Hard Sudoku Generalization ----
# Same models (trained on easy), evaluated on hard test set
print("\n" + "=" * 60)
print("TABLE 5: Hard Sudoku Generalization")
print("=" * 60)

table5_results = {}
NUM_EVAL_HARD = 1000

for strategy in ['vanilla', 'top_prob', 'top_prob_margin']:
    acc = evaluate_sudoku(
        sudoku_mdm, sudoku_test_hard_clues, sudoku_test_hard_solutions,
        strategy=strategy, model_type='mdm', num_eval=NUM_EVAL_HARD
    )
    table5_results[f'MDM_{strategy}'] = acc
    print(f"  MDM ({strategy}): {acc*100:.2f}%")

acc = evaluate_sudoku(
    sudoku_arm_with_order, sudoku_test_hard_clues, sudoku_test_hard_solutions,
    strategy='vanilla', model_type='arm', num_eval=NUM_EVAL_HARD
)
table5_results['ARM_with_order'] = acc
print(f"  ARM (with ordering): {acc*100:.2f}%")

with open(f'{BASE_DIR}/results/table5.json', 'w') as f:
    json.dump(table5_results, f, indent=2)

print("\nExpected (paper):")
print("  ARM (with ordering): 32.57%")
print("  MDM vanilla: 3.62%")
print("  MDM top_prob: 9.44%")
print("  MDM top_prob_margin: 49.88%")

---
## Cell 8: Train and Evaluate Zebra Puzzle Models (Table 3)

Same structure as Sudoku but:
- MDM: 19M params
- Different sequence length and vocabulary (from Shah et al. encoding)

In [ ]:
# Zebra dataset loading
# The exact format depends on Shah et al. (2024) codebase
# Adjust paths and loading code based on what you find in their repo

zebra_data_dir = f'{BASE_DIR}/data/zebra'
zebra_files = os.listdir(zebra_data_dir) if os.path.exists(zebra_data_dir) else []
print(f"Zebra data files: {zebra_files}")

if not zebra_files:
    print("\nWARNING: No Zebra data found.")
    print("Check Shah et al. (2024) repo for Zebra puzzle data.")
    print("The data format and vocabulary must match their codebase.")
    print("\nSkipping Zebra experiments for now.")
    ZEBRA_AVAILABLE = False
else:
    ZEBRA_AVAILABLE = True
    # Load Zebra data here — adjust based on actual format
    # zebra_train = np.load(f'{zebra_data_dir}/train.npy')
    # zebra_test = np.load(f'{zebra_data_dir}/test.npy')
    # ZEBRA_VOCAB = ...  # Determine from data
    # ZEBRA_SEQ_LEN = ...  # Determine from data

In [ ]:
if ZEBRA_AVAILABLE:
    print("=" * 60)
    print("TABLE 3: Zebra Puzzle Accuracy")
    print("=" * 60)
    
    # Train 19M MDM for Zebra
    zebra_mdm = MDMTransformer(
        vocab_size=ZEBRA_VOCAB, max_seq_len=ZEBRA_SEQ_LEN, **CONFIGS['19M']
    ).to(device)
    
    # Train 42M ARMs for Zebra
    # ... (same pattern as Sudoku)
    
    # Evaluate
    table3_results = {}
    # ... (same evaluation pattern as Sudoku)
    
    with open(f'{BASE_DIR}/results/table3.json', 'w') as f:
        json.dump(table3_results, f, indent=2)
else:
    print("Zebra experiments skipped — obtain data from Shah et al. (2024) repo first.")
    print("\nExpected results (paper):")
    print("  ARM (w/o ordering): 80.31%")
    print("  ARM (with ordering): 91.17%")
    print("  MDM vanilla: 76.9%")
    print("  MDM top_prob: 98.5%")
    print("  MDM top_prob_margin: 98.3%")

---
## Cell 9: Results Summary

In [ ]:
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

for table_name, results_file in [('Table 2', 'table2.json'), ('Table 5', 'table5.json'), ('Table 3', 'table3.json')]:
    path = f'{BASE_DIR}/results/{results_file}'
    if os.path.exists(path):
        with open(path) as f:
            results = json.load(f)
        print(f"\n{table_name}:")
        for method, acc in results.items():
            print(f"  {method}: {acc*100:.2f}%")
    else:
        print(f"\n{table_name}: not yet generated")